# TELUGU DEVOTEE — Wan2.1 free motion-video smoke test

This notebook tests **one short AI-generated motion clip** using the open Wan2.1 T2V 1.3B model. It does not upload to YouTube. Run in Google Colab with a GPU runtime if one is available.

**Free-tier caveat:** GPU access, session duration, disk space, and model download time are not guaranteed. If no GPU is assigned or the runtime runs out of memory, stop rather than switching to paid services.

In [ ]:
# Install inference libraries (may take several minutes)
%pip -q install -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg

In [ ]:
import torch, os
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
else:
    raise RuntimeError('No GPU is available in this runtime. In Colab, select Runtime > Change runtime type > T4 GPU (if offered), reconnect, then rerun.')

## 1. Enter one scene prompt

Use an **English visual prompt** for this first test (the model can be less reliable with long Telugu prompts). Keep the actual character dialogue/audio for the later editing stage; this notebook generates silent video only.

In [ ]:
prompt = '''Cinematic mythological scene from the Ramayana, young Prince Rama in saffron-yellow dhoti and yellow angavastram, shoulder-length black hair tied back, small vertical tilak, carrying a traditional bow and quiver, walking slowly through an ancient Indian forest beside sage Vishwamitra in saffron robes with long grey beard and wooden staff. Tall sal trees, morning mist, sun rays through leaves, floating dust motes, cloth and hair move gently in the breeze, slow cinematic tracking camera, consistent faces and anatomy, richly detailed hand-painted cinematic fantasy, natural motion, no text, no subtitles.'''
negative_prompt = '''blurry, low quality, distorted face, extra fingers, extra limbs, deformed hands, duplicated people, flicker, jitter, warped body, plastic skin, modern objects, text, watermark, subtitles, logo'''

## 2. Load the model and generate a short test

The first run downloads several gigabytes of model files. This cell may take a long time. It intentionally generates only 17 frames for a small smoke test; that is about one second at 16 fps, not a full scene.

In [ ]:
from diffusers import WanPipeline
from diffusers.utils import export_to_video

model_id = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
pipe = WanPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
try:
    pipe.vae.enable_tiling()
except Exception:
    pass

with torch.inference_mode():
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=480,
        width=832,
        num_frames=17,  # Wan frame count follows 4*k+1
        num_inference_steps=8,
        guidance_scale=5.0,
    )

frames = result.frames[0]
os.makedirs('/content/telugu_devotee_output', exist_ok=True)
output_path = '/content/telugu_devotee_output/wan_motion_test.mp4'
export_to_video(frames, output_path, fps=16)
print('Created:', output_path)

In [ ]:
# Preview and download the clip for your own review
from IPython.display import Video, display
display(Video(output_path, embed=True))

In [ ]:
from google.colab import files
files.download(output_path)

## Review checklist
- Check face/character consistency, hands, bow, cloth motion, flicker, and scene continuity.
- This output has **no Telugu voice, lip-sync, music, SEO packaging, or YouTube upload**.
- If the test works, the next engineering step is batching prompts from `output/episode_package.md`, saving each clip as a separate artifact, and keeping all clips in a manual-review stage.
- Do not put YouTube upload credentials in this notebook. Publishing must remain disabled until you explicitly approve the final video.